# Part 4 — GenAI: Turn Call-Centre Notes into Structured Reasons

CareLoop Clinics — No-Show Guard

Use Gemini API to classify follow-up notes, validate with Pydantic, test edge cases, and produce recommendation.

In [12]:
# Install required packages
!pip install -q google-genai pydantic

import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types

print("Imports loaded.")

Imports loaded.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Task 4.1 — One Validated Call

In [13]:
# Configuration
# Use the current Flash model from AI Studio (e.g., gemini-1.5-flash or gemini-2.0-flash)
MODEL_NAME = "gemini-3.5-flash-lite"  # <-- UPDATE to current Flash model from AI Studio

SYSTEM_TEXT = """You are a medical clinic operations analyst. Classify patient no-show reasons from call-centre notes into structured categories. Be precise and concise."""

CRAFT_PROMPT = """Context: CareLoop Clinics runs outpatient clinics. Patients miss appointments and call-centre staff write notes after phoning them.
Role: You are an analyst categorising no-show reasons.
Action: Read the note and classify the root cause into exactly one of these allowed values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other. If uncertain, choose 'other'.
Format: Return JSON with exactly these fields: root_cause (string), confidence (0-1), summary (max 25 words), recommended_action (string).
Tone: Professional, objective, no speculation."""

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"]
}

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

def analyse_note(note_text, client, prompt_template=CRAFT_PROMPT):
    """Send one note to Gemini and return (parsed_dict, raw_text)."""
    prompt = f"{prompt_template}\n\nNote: {note_text}"
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")  # DELETE this line if model rejects it
        )
    )
    return json.loads(response.text), response.text

print("Configuration ready.")

Configuration ready.


In [14]:
# Get API key at runtime (never hardcode, never print)
API_KEY = input("Paste your Gemini API key: ").strip()
client = genai.Client(api_key=API_KEY)

# Test on N001
notes_df = pd.read_csv("data/followup_notes.csv")
n001 = notes_df[notes_df["note_id"] == "N001"]["note_text"].values[0]
print(f"N001 note: {n001}")

result, raw = analyse_note(n001, client)
print(f"\nParsed result:")
print(json.dumps(result, indent=2))

N001 note: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.

Parsed result:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 1.0,
  "summary": "Patient missed the appointment because she did not receive a reminder and forgot.",
  "recommended_action": "Reschedule appointment and verify patient contact details for future reminders."
}


## Task 4.2 — Validate and Process 10 Notes

In [15]:
# Pydantic model
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def validate_insight(data):
    """Validate with Pydantic and allowed-list check."""
    try:
        insight = FollowupInsight(**data)
        if insight.root_cause not in ALLOWED_CAUSES:
            return False, "root_cause not in allowed list"
        return True, "valid"
    except ValidationError as e:
        return False, str(e)

# Process first 10 notes
first_10 = notes_df.head(10).copy()

results = []
api_calls = 0

for _, row in first_10.iterrows():
    note_id = row["note_id"]
    appt_id = row["appointment_id"]
    note_text = row["note_text"]
    
    print(f"Processing {note_id} (appt {appt_id})...")
    
    # Check for blank note
    if pd.isna(note_text) or str(note_text).strip() == "":
        result = {
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
        }
        validation_status = "skipped_blank"
        raw_response = ""
        print(f"  -> Skipped (blank)")
    else:
        try:
            api_result, raw_response = analyse_note(str(note_text), client)
            api_calls += 1
            is_valid, msg = validate_insight(api_result)
            validation_status = "valid" if is_valid else "invalid"
            result = api_result
            # raw_response already contains the exact raw text from API
            print(f"  -> {validation_status}: {result.get('root_cause', 'N/A')}")
        except Exception as e:
            validation_status = "invalid"
            result = {}
            raw_response = f"ERROR: {e}"
            print(f"  -> ERROR: {e}")
    
    # Build output row: copy only fields that exist in result, leave missing empty
    row_out = {
        "note_id": note_id,
        "appointment_id": appt_id,
        "root_cause": result.get("root_cause", ""),
        "confidence": result.get("confidence", ""),
        "summary": result.get("summary", ""),
        "recommended_action": result.get("recommended_action", ""),
        "validation_status": validation_status,
        "raw_response": raw_response
    }
    # For skipped_blank, fill the defaults
    if validation_status == "skipped_blank":
        row_out["confidence"] = 0.0
        row_out["summary"] = "No note recorded"
        row_out["recommended_action"] = "Re-call patient"
    
    results.append(row_out)
    
    # Rate limit: 13 seconds between calls
    if _ < len(first_10) - 1:
        time.sleep(13)

# Save
notes_analysed = pd.DataFrame(results)
notes_analysed.to_csv("notes_analysed.csv", index=False)
print(f"\nSaved notes_analysed.csv: {notes_analysed.shape}")
print(f"Total API calls so far: {api_calls}")
print(notes_analysed[['note_id', 'root_cause', 'validation_status']].to_string(index=False))

Processing N001 (appt A0872)...
  -> valid: forgot_or_no_reminder
Processing N002 (appt A0897)...
  -> valid: distance_or_transport
Processing N003 (appt A0526)...
  -> valid: cost_or_fee
Processing N004 (appt A0926)...
  -> Skipped (blank)
Processing N005 (appt A0534)...
  -> valid: illness_or_emergency
Processing N006 (appt A0507)...
  -> valid: work_or_schedule
Processing N007 (appt A0333)...
  -> valid: forgot_or_no_reminder
Processing N008 (appt A0222)...
  -> ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing N009 (appt A0977)...
  -> valid: other
Processing N010 (appt A0596)...
  -> valid: work_or_schedule

Saved notes_analysed.csv: (10, 8)
Total API calls so far: 8
note_id            root_cause validation_status
   N001 forgot_or_no_reminder             valid
   N002 distance_or_transport             valid
   N003     

## Task 4.3 — Break It on Purpose

In [16]:
# (a) Pydantic validation tests
print("--- Pydantic Validation Tests ---")

# Test 1: confidence = 1.7 (out of bounds)
try:
    FollowupInsight(root_cause="other", confidence=1.7, summary="test", recommended_action="test")
except ValidationError as e:
    print(f"confidence=1.7 error: {e}")

# Test 2: missing recommended_action
try:
    FollowupInsight(root_cause="other", confidence=0.5, summary="test")
except ValidationError as e:
    print(f"missing recommended_action error: {e}")

--- Pydantic Validation Tests ---
confidence=1.7 error: 3 validation errors for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
summary
  String should have at least 5 characters [type=string_too_short, input_value='test', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short
recommended_action
  String should have at least 5 characters [type=string_too_short, input_value='test', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short
missing recommended_action error: 2 validation errors for FollowupInsight
summary
  String should have at least 5 characters [type=string_too_short, input_value='test', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short
recommended_action
  Field requi

In [17]:
# (b) Prompt comparison for N007 and N009
print("--- Prompt Comparison (V1 vs V2) ---")

V1_PROMPT = "Classify this note."

comparison_notes = [
    ("N007", notes_df[notes_df["note_id"]=="N007"]["note_text"].values[0]),
    ("N009", notes_df[notes_df["note_id"]=="N009"]["note_text"].values[0]),
]

comparison_results = []

for note_id, note_text in comparison_notes:
    for prompt_name, prompt_template in [("V1", V1_PROMPT), ("V2", CRAFT_PROMPT)]:
        try:
            result, raw = analyse_note(note_text, client, prompt_template)
            api_calls += 1
            root_cause = result.get("root_cause", "")
            allowed = root_cause in ALLOWED_CAUSES
            comparison_results.append({
                "note_id": note_id,
                "prompt": prompt_name,
                "root_cause": root_cause,
                "allowed": allowed
            })
            print(f"  {note_id} | {prompt_name} | {root_cause} | allowed={allowed}")
            time.sleep(13)
        except Exception as e:
            comparison_results.append({
                "note_id": note_id,
                "prompt": prompt_name,
                "root_cause": f"ERROR: {e}",
                "allowed": False
            })
            print(f"  {note_id} | {prompt_name} | ERROR: {e}")

comp_df = pd.DataFrame(comparison_results)
print(f"\n{comp_df.to_string(index=False)}")

v2_allowed = comp_df[(comp_df["prompt"]=="V2")]["allowed"].all()
print(f"\nDeploy V2 (CRAFT prompt)? {v2_allowed}")

--- Prompt Comparison (V1 vs V2) ---
  N007 | V1 | Patient forgot and did not receive SMS reminder | allowed=False
  N007 | V2 | ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
  N009 | V1 | unreachable | allowed=False
  N009 | V2 | other | allowed=True

note_id prompt                                                                                                                                                                                                root_cause  allowed
   N007     V1                                                                                                                                                           Patient forgot and did not receive SMS reminder    False
   N007     V2 ERROR: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in dema

In [18]:
# (c) Total API calls count
print(f"\n--- Total API Calls ---")
print(f"Total API calls used: {api_calls}")


--- Total API Calls ---
Total API calls used: 11


## Task 4.4 — Recommendation and GitHub

In [19]:
# Load results from Part 3 and notes_analysed
with open("results_part3.json", "r") as f:
    p3 = json.load(f)

appt = pd.read_csv("appointments_clean.csv")
notes = pd.read_csv("notes_analysed.csv")

merged = notes.merge(appt[['appointment_id', 'reminder_sent']], on='appointment_id', how='left')

# Cross-tabulation
crosstab = pd.crosstab(merged['root_cause'], merged['reminder_sent'])
print("Cross-tab: root_cause vs reminder_sent")
print(crosstab)

# Key numbers from actual outputs
print(f"\n--- Key Numbers for README ---")
print(f"Part 1: 990 clean appointments, 22.0% no-show rate")
print(f"Part 2: Highest no-show clinic Gomti Nagar 33.3% (90 appts); lead_days 31+ band 60.5% no-show")
print(f"Part 3: {p3['winner']} chosen (CV AUC={p3['cv_auc']:.3f}), threshold={p3['threshold']}, test recall={p3['test_recall']:.2f}")
print(f"Part 4: {len(merged)} notes analysed; {merged['validation_status'].value_counts().get('valid', 0)} valid")

if 'forgot_or_no_reminder' in merged['root_cause'].values:
    forgot = merged[merged['root_cause']=='forgot_or_no_reminder']
    print(f"forgot_or_no_reminder notes: {len(forgot)} total, reminder_sent=No: {(forgot['reminder_sent']=='No').sum()}")

print(f"Notes with reminder_sent=No: {(merged['reminder_sent']=='No').sum()} out of {len(merged)}")

Cross-tab: root_cause vs reminder_sent
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    0
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1

--- Key Numbers for README ---
Part 1: 990 clean appointments, 22.0% no-show rate
Part 2: Highest no-show clinic Gomti Nagar 33.3% (90 appts); lead_days 31+ band 60.5% no-show
Part 3: Logistic Regression chosen (CV AUC=0.774), threshold=0.35, test recall=0.57
Part 4: 10 notes analysed; 8 valid
forgot_or_no_reminder notes: 2 total, reminder_sent=No: 2
Notes with reminder_sent=No: 5 out of 10


In [20]:
# Generate README.md from actual results
winner = p3['winner']
cv_auc = p3['cv_auc']
threshold = p3['threshold']
test_prec = p3['test_precision']
test_rec = p3['test_recall']
test_flagged = p3['test_flagged']
test_auc = p3['test_auc']
lr_cv_auc = p3['lr_cv_auc']
dt3_cv_auc = p3['dt3_cv_auc']
rf_cv_auc = p3['rf_cv_auc']
rf_tuned_cv_auc = p3['rf_tuned_cv_auc']

valid_count = int(merged['validation_status'].value_counts().get('valid', 0))
total_notes = len(merged)
forgot_count = int((merged['root_cause'] == 'forgot_or_no_reminder').sum())
no_reminder_count = int((merged['reminder_sent'] == 'No').sum())

readme = f"""# CareLoop Clinics — No-Show Guard

https://github.com/<PASTE_REPO_LINK>

## Summary (One Finding Per Part)

1. **Part 1 (Data Foundations):** Cleaned 1,020 raw bookings into 990 valid appointments; 22.0% overall no-show rate; outliers capped (26 distance, 3 fee).
2. **Part 2 (SQL Analysis):** No-shows cluster at Gomti Nagar (33.3%, 90 appts) and rise sharply with lead time — 60.5% no-show rate for 31+ day lead.
3. **Part 3 (ML Model):** {winner} selected (CV ROC-AUC = {cv_auc:.3f}); threshold {threshold} gives {test_rec:.0%} recall on test set, flagging {test_flagged} risky bookings.
4. **Part 4 (GenAI Notes):** Of {total_notes} call-centre notes analysed, {valid_count} valid; top reasons from notes: forgot/no reminder, distance, work conflicts.

## Recommendation

CareLoop should prioritise **automated reminders for bookings with lead time >14 days** (29.9% of appointments, 30.0% no-show rate in 15-30 day band, 60.5% in 31+ band) and **targeted follow-up calls for patients with prior no-shows** (lead_days and prior_no_shows both have strong positive coefficients in the model). The model flags {test_flagged} high-risk appointments per {p3['test_size']} test cases at threshold {threshold}, with {test_rec:.0%} recall — meaning over half of actual no-shows are caught. GenAI analysis confirms that {forgot_count} of {total_notes} notes cite forgotten appointments, and {no_reminder_count} of {total_notes} analysed notes had no reminder sent. Deploying reminders for all bookings >7 days out and phoning the top-{test_flagged} risk-scored patients each evening would address the largest avoidable causes.

## Model Chosen

**{winner}** (5-fold CV ROC-AUC = {cv_auc:.3f}) — highest among all candidates (Depth-3 Tree: {dt3_cv_auc:.3f}, Random Forest: {rf_cv_auc:.3f}, Tuned RF: {rf_tuned_cv_auc:.3f}). Threshold **{threshold}** chosen via out-of-fold F1 optimisation, yielding test recall={test_rec:.2f}, precision={test_prec:.2f}, flagging {test_flagged} appointments.

## Limitations

1. The {total_notes} analysed notes are an illustration and cannot show real clinic-wide reasons.
2. Cleaning statistics (medians, quartiles, outlier fences) were computed on all 990 rows before the train/test split, giving test rows indirect influence.
3. The same patient can appear in both training and test sets (random split), and the split is random rather than past-to-future.

---
*All code in this repository uses only free tools: Python (pandas, scikit-learn), SQL (MySQL/SQLite), and Gemini API (free tier).*
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("README.md generated.")

README.md generated.
